# 9 · Justificativas do BC para o juro alto: o que os dados mostram (2012–2026)

**Pergunta:** as justificativas usadas pelo Banco Central para sustentar o
juro alto — inflação acima da meta, mercado de trabalho aquecido, rendimentos
crescendo acima da produtividade, expectativas desancoradas, choques de
oferta — se sustentam nos dados?

**O que este notebook FAZ:**

1. compara o IPCA com a **meta efetiva de cada ano** (não com a simplificação
   "4,5% até 2024");
2. decompõe o IPCA 12m por grupo do IPCA (aproximação declarada, com resíduo);
3. estima curvas de Phillips simples para o IPCA cheio e para serviços, com
   erro-padrão HAC — **associação condicional, não causalidade**;
4. mede salário real, produtividade por trabalhador e ULC;
5. descreve o mercado de trabalho (desemprego, ocupação, participação,
   informalidade);
6. lê o Focus (base de cálculo 0) e as atas 280–282, com citação verbatim;
7. fecha com a tabela **"o que o BC diz / o que os dados mostram"**.

**O que este notebook NÃO faz:**

- **não** identifica causalidade: a Phillips é correlação condicional; a
  decomposição é contábil; o juro alto não é tratado como causa única;
- **não** mede produtividade por hora (a PNAD agregada não publica horas no
  SIDRA) — usa produtividade por trabalhador;
- **não** transforma discurso ou entrevista em dado: as atas são citadas
  verbatim; entrevistas só entrariam com transcrição literal;
- **não** conclui qual é o "prêmio institucional" — apenas mostra o que os
  dados permitem e o que não permitem.

**Fontes:** SGS/BCB (IPCA, serviços, câmbio), SIDRA/IBGE (IPCA por grupos,
PNAD Contínua, PIB volume), Focus/Olinda (base 0), atas do Copom 280–282.
Tudo em `dados/processado/`; nenhum download neste notebook.

In [1]:
\
import sys
import textwrap
from pathlib import Path

p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, MARROM, CINZA,
)
from src.processa import justificativas as j

estilo()
print("dados:", p / "dados" / "processado")

dados: /home/vanessa/devel/juros_e_divida/dados/processado


## 1 · Metas efetivas, faixa de tolerância e IPCA realizado

A meta não foi 4,5% o tempo todo: ela caiu para 4,25% em 2019, 4,00% em 2020, 3,75% em 2021, 3,50% em 2022, 3,25% em 2023 e 3,00% em 2024–26. A tolerância também mudou (±2,0 p.p. até 2016; ±1,5 depois). Comparar o IPCA com uma meta fixa de 4,5% esconde os anos em que a meta era mais baixa.

In [2]:
\
m = j.metas_anuais()
tab = m[["ano", "meta_pct", "tolerancia_pp", "piso_pct", "teto_pct",
         "ipca_pct", "fechado", "dentro", "abaixo_piso", "acima_teto"]].rename(
    columns={
        "ano": "ano",
        "meta_pct": "meta (%)",
        "tolerancia_pp": "± (p.p.)",
        "piso_pct": "piso (%)",
        "teto_pct": "teto (%)",
        "ipca_pct": "IPCA (%)",
        "fechado": "ano fechado?",
        "dentro": "dentro?",
        "abaixo_piso": "abaixo do piso?",
        "acima_teto": "acima do teto?",
    })
display(tab)
print("2026: a coluna IPCA é o 12m de out/26, não o acumulado no ano.")

,ano,meta (%),± (p.p.),piso (%),teto (%),IPCA (%),ano fechado?,dentro?,abaixo do piso?,acima do teto?
0,2012,4.50,2.00,2.50,6.50,5.84,True,True,False,False
1,2013,4.50,2.00,2.50,6.50,5.91,True,True,False,False
2,2014,4.50,2.00,2.50,6.50,6.41,True,True,False,False
3,2015,4.50,2.00,2.50,6.50,10.67,True,False,False,True
4,2016,4.50,2.00,2.50,6.50,6.29,True,True,False,False
5,2017,4.50,1.50,3.00,6.00,2.95,True,False,True,False
6,2018,4.50,1.50,3.00,6.00,3.75,True,True,False,False
7,2019,4.25,1.50,2.75,5.75,4.31,True,True,False,False
8,2020,4.00,1.50,2.50,5.50,4.52,True,True,False,False
9,2021,3.75,1.50,2.25,5.25,10.06,True,False,False,True


2026: a coluna IPCA é o 12m de out/26, não o acumulado no ano.


In [3]:
\
sgs = pd.read_parquet(p / "dados" / "processado" / "sgs.parquet")
ipca12 = (sgs[sgs.serie == "ipca_12m"]
          .assign(data=lambda d: pd.to_datetime(d["data"]))
          .set_index("data")["valor"].sort_index())
ipca12 = ipca12[ipca12.index >= "2012-01-01"]

meta = pd.Series([j.METAS_ANUAIS[a][0] for a in ipca12.index.year],
                 index=ipca12.index)
tol = pd.Series([j.METAS_ANUAIS[a][1] for a in ipca12.index.year],
                index=ipca12.index)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.fill_between(ipca12.index, meta - tol, meta + tol, color=CINZA,
                alpha=0.15, label="faixa de tolerância")
ax.plot(meta.index, meta, color=CINZA, ls="--", lw=1.2, label="meta")
ax.plot(ipca12.index, ipca12, color=AZUL, lw=1.8, label="IPCA 12m")

dez = ipca12[ipca12.index.month == 12]
cores = []
for d, v in dez.items():
    mm, tt = j.METAS_ANUAIS[d.year]
    cores.append(VERMELHO if v > mm + tt else (LARANJA if v < mm - tt else VERDE))
ax.scatter(dez.index, dez.values, c=cores, s=28, zorder=5, edgecolor="white")
for cor, rotulo in ((VERMELHO, "acima do teto"), (LARANJA, "abaixo do piso"),
                    (VERDE, "dentro")):
    ax.scatter([], [], color=cor, s=28, label=rotulo)
if ipca12.index.max().year == 2026:
    ax.annotate(f"out/26: {ipca12.iloc[-1]:.2f}%",
                xy=(ipca12.index[-1], ipca12.iloc[-1]),
                xytext=(-70, 22), textcoords="offset points", fontsize=9,
                arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))
ax.set_ylabel("% em 12 meses")
ax.set_title("IPCA 12m contra a meta efetiva e a faixa de tolerância")
ax.legend(fontsize=9, ncol=3, loc="upper right")
ax.margins(x=0.01)
salvar(fig, "09_metas")

fig -> viz/figs/09_metas.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_metas.png')

**Figura:** `viz/figs/09_metas.png` — a linha azul é o IPCA 12m; a faixa cinza é a tolerância do ano; os pontos de dezembro são coloridos pelo veredito (vermelho acima do teto, laranja abaixo do piso, verde dentro).

## 2 · O que compõe o IPCA: grupos, não só salário

A contribuição é `variação 12m do grupo × peso do mês`, uma aproximação padrão. A soma dos 9 grupos difere do IPCA cheio em até 0,36 p.p. (peso muda ao longo dos 12 meses); o resíduo está na tabela. O proxy de **serviços** (grupos 6–9: saúde, despesas pessoais, educação, comunicação) **não** é a série oficial de serviços do BCB (SGS 10844) — as duas aparecem lado a lado.

In [4]:
\
c = j.contribuicoes_ipca()
s = j.servicos_intensivos()

serv = (sgs[sgs.serie == "ipca_servicos"]
        .assign(data=lambda d: pd.to_datetime(d["data"]))
        .set_index("data")["valor"].sort_index())
serv12 = ((1 + serv / 100).rolling(12).apply(np.prod, raw=True) - 1) * 100

def _serv12(ano):
    ref = (pd.Timestamp(f"{ano}-12-31") if ano < 2026
           else pd.Timestamp("2026-10-31"))
    return float(serv12.asof(ref))

linhas = []
for ano in range(2021, 2027):
    d = c[c.ano == ano].sort_values("contribuicao_pp", ascending=False)
    ss = s[s.ano == ano].iloc[0]
    top = d.head(3)
    linhas.append({
        "ano": ano,
        "IPCA 12m (%)": d.total_ipca_12m.iloc[0],
        "1º grupo": f"{top.iloc[0].grupo} ({top.iloc[0].contribuicao_pp:.2f})",
        "2º grupo": f"{top.iloc[1].grupo} ({top.iloc[1].contribuicao_pp:.2f})",
        "3º grupo": f"{top.iloc[2].grupo} ({top.iloc[2].contribuicao_pp:.2f})",
        "serviços 6–9 (p.p.)": ss.contrib_servicos_pp,
        "share dos serviços (%)": ss.share_pct,
        "SGS 10844 12m (%)": _serv12(ano),
        "resíduo (p.p.)": d.residuo_pp.iloc[0],
    })
display(pd.DataFrame(linhas))

,ano,IPCA 12m (%),1º grupo,2º grupo,3º grupo,serviços 6–9 (p.p.),share dos serviços (%),SGS 10844 12m (%),resíduo (p.p.)
0,2021,10.06,Transportes (4.61),Habitação (2.11),Alimentação e bebidas (1.64),1.16,11.49,4.75,0.36
1,2022,5.79,Alimentação e bebidas (2.54),Saúde e cuidados pessoais (1.48),Vestuário (0.87),2.63,45.49,7.57,0.31
2,2023,4.62,Transportes (1.50),Saúde e cuidados pessoais (0.88),Habitação (0.78),2.05,44.35,6.22,0.07
3,2024,4.83,Alimentação e bebidas (1.66),Saúde e cuidados pessoais (0.82),Transportes (0.68),1.88,38.93,4.74,0.03
4,2025,4.26,Habitação (1.05),Saúde e cuidados pessoais (0.76),Alimentação e bebidas (0.63),1.78,41.75,6.00,0.05
5,2026,4.63,Alimentação e bebidas (1.23),Saúde e cuidados pessoais (0.74),Transportes (0.65),1.88,40.56,5.65,-0.01


In [5]:
\
anos = [2021, 2022, 2023, 2024, 2025, 2026]
piv = (c[c.ano.isin(anos)]
       .pivot(index="ano", columns="grupo", values="contribuicao_pp")
       .reindex(anos))
ordem = piv.sum().sort_values(ascending=False).index
piv = piv[ordem]
cores = [AZUL, VERMELHO, VERDE, LARANJA, ROXO, MARROM, CINZA,
         "#bcbd22", "#17becf"]
fig, ax = plt.subplots(figsize=(11, 5.5))
left = np.zeros(len(piv))
for grupo, cor in zip(piv.columns, cores):
    ax.barh(piv.index.astype(str), piv[grupo], left=left, color=cor,
            label=grupo)
    left = left + piv[grupo].fillna(0).values
for i, ano in enumerate(piv.index):
    total = c[c.ano == ano].total_ipca_12m.iloc[0]
    ax.text(left[i] + 0.08, i, f"soma {left[i]:.2f} | IPCA {total:.2f}",
            va="center", fontsize=8)
ax.set_xlabel("contribuição (p.p.)")
ax.set_title("Contribuição por grupo para o IPCA 12m — soma aproximada")
ax.legend(fontsize=7, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.12))
ax.margins(x=0.18)
salvar(fig, "09_contribuicoes")

fig -> viz/figs/09_contribuicoes.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_contribuicoes.png')

**Figura:** `viz/figs/09_contribuicoes.png` — barras empilhadas por grupo; o texto à direita mostra a soma das contribuições e o IPCA cheio. 2026 = 12m até out/26.

## 3 · Phillips: desemprego e inflação, associação condicional

Modelos OLS com erro-padrão HAC (12 lags). A variável dependente é o IPCA 12m (ou serviços 12m); a explicativa de interesse é o desemprego. Focus 12m e câmbio 12m entram como controles. **Sinal negativo** é o previsto pela curva de Phillips; **positivo** é o oposto. Nenhum modelo identifica causalidade.

In [6]:
\
ph = j.phillips()
cols = ["modelo", "coef", "erro_padrao", "p_valor", "r2", "n", "janela"]
print("Amostra 2012–2026")
display(ph[ph.variavel == "u"][cols].rename(columns={
    "modelo": "modelo", "coef": "coef. u", "erro_padrao": "erro-padrão",
    "p_valor": "p", "r2": "R²", "n": "n", "janela": "janela"}))
print("Amostra 2021–2026")
phr = j.phillips(("2021-01", "2026-09"))
display(phr[phr.variavel == "u"][cols].rename(columns={
    "modelo": "modelo", "coef": "coef. u", "erro_padrao": "erro-padrão",
    "p_valor": "p", "r2": "R²", "n": "n", "janela": "janela"}))

Amostra 2012–2026


,modelo,coef. u,erro-padrão,p,R²,n,janela
1,IPCA cheio ~ desemprego,-0.05,0.15,0.73,0.00,175,2012-03–2026-09
3,IPCA cheio ~ desemprego + Focus + câmbio,0.20,0.12,0.08,0.59,175,2012-03–2026-09
7,Serviços ~ desemprego,-0.50,0.15,0.00,0.42,175,2012-03–2026-09
9,Serviços ~ desemprego + Focus + câmbio,-0.31,0.10,0.00,0.80,175,2012-03–2026-09


Amostra 2021–2026


,modelo,coef. u,erro-padrão,p,R²,n,janela
1,IPCA cheio ~ desemprego,0.53,0.20,0.01,0.37,69,2021-01–2026-09
3,IPCA cheio ~ desemprego + Focus + câmbio,0.41,0.07,0.00,0.71,69,2021-01–2026-09
7,Serviços ~ desemprego,-0.31,0.15,0.04,0.26,69,2021-01–2026-09
9,Serviços ~ desemprego + Focus + câmbio,-0.39,0.09,0.00,0.64,69,2021-01–2026-09


In [7]:
\
b = j.base_phillips().copy()
b.index = b.index.to_timestamp(how="end")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y, titulo in ((axes[0], "ipca12", "IPCA cheio"),
                      (axes[1], "serv12", "Serviços")):
    pre = b[b.index < "2020-01-01"]
    mid = b[(b.index >= "2020-01-01") & (b.index < "2021-01-01")]
    pos = b[b.index >= "2021-01-01"]
    ax.scatter(pre["u"], pre[y], s=14, color=AZUL, alpha=0.55,
               label="2012–2019")
    ax.scatter(mid["u"], mid[y], s=14, color=CINZA, alpha=0.55, label="2020")
    ax.scatter(pos["u"], pos[y], s=14, color=VERMELHO, alpha=0.55,
               label="2021–2026")
    coef = np.polyfit(b["u"], b[y], 1)
    xs = np.linspace(b["u"].min(), b["u"].max(), 60)
    ax.plot(xs, np.polyval(coef, xs), color="black", lw=1.5,
            label="ajuste linear (amostra toda)")
    ax.set_xlabel("desemprego (%)")
    ax.set_ylabel(f"{titulo} 12m (%)")
    ax.set_title(f"{titulo} × desemprego — associação, não causalidade")
    ax.legend(fontsize=8)
fig.tight_layout()
salvar(fig, "09_phillips")

fig -> viz/figs/09_phillips.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_phillips.png')

**Figura:** `viz/figs/09_phillips.png` — cada ponto é um mês; a linha preta é o ajuste linear da amostra toda. O contraste visual entre 2012–19 (azul) e 2021–26 (vermelho) mostra por que o sinal do IPCA cheio se inverte no período recente.

## 4 · Salário real, produtividade e ULC

Produtividade = PIB volume (CNT) ÷ pessoas ocupadas (PNAD), média anual. É produtividade **por trabalhador**, não por hora. ULC = variação do salário real − variação da produtividade (proxy; não é a ULC oficial do IBGE).

In [8]:
\
t = j.produtividade_anual()
cols = ["ano", "sal_yoy_pct", "prod_yoy_pct", "ulc_pp", "razao_sal_prod",
        "ipca_dez_pct", "servicos_dez_pct", "desemprego_medio_pct"]
display(t[(t.ano >= 2012) & (t.ano <= 2025)][cols].rename(columns={
    "ano": "ano",
    "sal_yoy_pct": "salário real (%)",
    "prod_yoy_pct": "produtividade (%)",
    "ulc_pp": "ULC (p.p.)",
    "razao_sal_prod": "razão salário/prod.",
    "ipca_dez_pct": "IPCA dez (%)",
    "servicos_dez_pct": "serviços dez (%)",
    "desemprego_medio_pct": "desemprego médio (%)",
}))
print("2012 é o ano-base (sem variação); 2026 ainda incompleto.")

,ano,salário real (%),produtividade (%),ULC (p.p.),razão salário/prod.,IPCA dez (%),serviços dez (%),desemprego médio (%)
0,2012,NaN,NaN,NaN,1.00,5.84,8.74,7.36
1,2013,3.31,1.53,1.77,1.02,5.91,8.75,7.26
2,2014,2.98,-0.67,3.65,1.05,6.41,8.33,6.86
3,2015,-3.93,-3.34,-0.59,1.05,10.67,8.11,8.40
4,2016,0.41,-1.11,1.52,1.06,6.29,6.48,11.37
5,2017,-0.54,1.22,-1.76,1.05,2.95,4.55,12.88
6,2018,2.22,0.28,1.95,1.07,3.75,3.35,12.37
7,2019,-0.40,-0.78,0.37,1.07,4.31,3.53,12.03
8,2020,3.31,5.12,-1.81,1.05,4.52,1.73,13.48
9,2021,-6.31,-0.15,-6.16,0.99,10.06,4.75,13.50


2012 é o ano-base (sem variação); 2026 ainda incompleto.


In [9]:
\
tt = t[t.ano <= 2025]
tu = t[t.ulc_pp.notna()]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(tt.ano, tt.sal_idx, color=VERMELHO, marker="o", lw=2,
         label="salário real")
ax1.plot(tt.ano, tt.prod_idx, color=AZUL, marker="o", lw=2,
         label="produtividade por trabalhador")
ax1.axhline(100, color=CINZA, lw=0.8)
ax1.set_title("2012 = 100: salário real × produtividade")
ax1.set_ylabel("índice")
ax1.legend(fontsize=9)
ax2.bar(tu.ano, tu.ulc_pp,
        color=[VERMELHO if v > 0 else AZUL for v in tu.ulc_pp])
ax2.axhline(0, color="k", lw=1)
ax2.set_title("ULC (salário real − produtividade, p.p.)")
ax2.set_ylabel("p.p.")
for ax in (ax1, ax2):
    ax.margins(x=0.03)
fig.tight_layout()
salvar(fig, "09_produtividade")

fig -> viz/figs/09_produtividade.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_produtividade.png')

**Figura:** `viz/figs/09_produtividade.png` — à esquerda, os índices (2012=100); à direita, a ULC anual. O salário real só recupera o nível de 2012 em 2023 e o gap acumulado chega a 15% em 2025.

## 5 · Mercado de trabalho: apertado, mas com menos oferta

Desemprego mensal (PNAD, 3 meses móveis) e indicadores trimestrais (ocupação, participação, informalidade). A informalidade só é publicada a partir de 2015Q4.

In [10]:
\
mt = j.mercado_trabalho_resumo()
display(mt.rename(columns={
    "indicador": "indicador",
    "ultimo": "último",
    "data_ultimo": "data",
    "media_2012_2019": "média 2012–19",
    "media_2025_2026": "média 2025–26",
    "minimo": "mínimo",
    "data_minimo": "data do mínimo",
    "maximo": "máximo",
    "data_maximo": "data do máximo",
}))

,indicador,último,data,média 2012–19,média 2025–26,mínimo,data do mínimo,máximo,data do máximo
0,Desemprego (%),5.20,2026-09-30,9.86,5.74,5.10,2025-12-31,14.90,2020-09-30
1,Nível de ocupação (%),59.00,2026-09-30,56.60,58.60,48.90,2020-09-30,59.00,2026-09-30
2,Taxa de participação (%),62.20,2026-09-30,62.81,62.17,57.10,2020-06-30,63.60,2019-06-30
3,Informalidade (%),37.50,2026-09-30,39.85,37.63,36.60,2020-06-30,41.00,2019-09-30
4,Pessoas ocupadas (mil),"103,638.00",2026-09-30,"90,719.38","102,418.43","81,954.00",2020-09-30,"103,638.00",2026-09-30


In [11]:
\
mtw = pd.read_parquet(p / "dados" / "processado" / "mercado_trabalho.parquet")
piv = mtw.pivot(index="data", columns="variavel", values="valor").sort_index()
des = pd.read_parquet(p / "dados" / "processado" / "desemprego.parquet")
u = (des.assign(data=lambda d: pd.to_datetime(d["data"]))
     .set_index("data")["taxa_pct"].sort_index())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(u.index, u.values, color=AZUL, lw=1.1, label="desemprego (mensal)")
ax1.axhline(u[u.index.year <= 2019].mean(), color=CINZA, ls="--", lw=1.2,
            label="média 2012–19")
ax1.set_title("Desemprego: mínimo da série em 2025–26")
ax1.set_ylabel("%")
ax1.legend(fontsize=9)
ax2.plot(piv.index, piv["nivel_ocupacao_pct"], color=VERDE, lw=2,
         label="nível de ocupação")
ax2.plot(piv.index, piv["participacao_pct"], color=LARANJA, lw=2,
         label="participação")
ax2.plot(piv.index, piv["informalidade_pct"], color=ROXO, lw=2,
         label="informalidade")
ax2.set_title("Ocupação em recorde; participação abaixo de 2012–19")
ax2.set_ylabel("%")
ax2.legend(fontsize=9)
fig.tight_layout()
salvar(fig, "09_trabalho")

fig -> viz/figs/09_trabalho.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_trabalho.png')

**Figura:** `viz/figs/09_trabalho.png` — à esquerda, o desemprego contra a média de 2012–19; à direita, ocupação, participação e informalidade. A ocupação está em recorde; a participação está abaixo da média histórica.

## 6 · Expectativas: Focus 12m e a meta em vigor

Série não suavizada e **base de cálculo 0** (a base 1 coexiste desde 2014 e não pode ser misturada). A comparação de credibilidade usa a meta efetiva de cada ano — não a meta de 3% atual, que só vale desde 2025.

In [12]:
\
e = j.expectativas_anuais()
display(e.rename(columns={
    "data_pesquisa": "pesquisa",
    "data_referencia": "ano",
    "media": "média (%)",
    "mediana": "mediana (%)",
    "desvio_padrao": "desvio-padrão (p.p.)",
    "meta_pct": "meta (%)",
    "teto_pct": "teto (%)",
    "acima_meta": "acima da meta?",
    "acima_teto": "acima do teto?",
}))
cred = j.credibilidade().rename(columns={
    "periodo": "período",
    "meses": "meses",
    "focus_12m_medio_pct": "Focus 12m médio (%)",
    "meta_media_pct": "meta média (%)",
    "gap_medio_pp": "gap médio (p.p.)",
    "dispersao_media_pp": "dispersão média (p.p.)",
    "pct_meses_acima_meta": "% meses acima da meta",
    "pct_meses_acima_teto": "% meses acima do teto",
})
display(cred.style.format({
    "meses": "{:.0f}",
    "Focus 12m médio (%)": "{:.3f}",
    "meta média (%)": "{:.2f}",
    "gap médio (p.p.)": "{:.2f}",
    "dispersão média (p.p.)": "{:.2f}",
    "% meses acima da meta": "{:.2f}",
    "% meses acima do teto": "{:.2f}",
}))

,pesquisa,ano,média (%),mediana (%),desvio-padrão (p.p.),meta (%),teto (%),acima da meta?,acima do teto?
0,2026-11-13,2026,5.11,5.11,0.15,3.00,4.50,True,True
1,2026-11-13,2027,4.96,5.00,0.60,3.00,4.50,True,True
2,2026-11-13,2028,4.05,4.00,0.51,3.00,4.50,True,False
3,2026-11-13,2029,3.89,4.00,0.53,3.00,4.50,True,False
4,2026-11-13,2030,3.86,3.94,0.56,3.00,4.50,True,False


,período,meses,Focus 12m médio (%),meta média (%),gap médio (p.p.),dispersão média (p.p.),% meses acima da meta,% meses acima do teto
0,2012–2019,96,5.235,4.47,0.77,0.38,66.67,11.46
1,2021–2026,71,4.633,3.25,1.38,0.46,98.59,39.44


In [13]:
\
foc = pd.read_parquet(p / "dados" / "processado" / "focus_inflacao12m.parquet")
foc = foc[(foc.suavizada == "N") & (foc.base_calculo == 0)].copy()
foc["data_pesquisa"] = pd.to_datetime(foc["data_pesquisa"])
foc = foc.sort_values("data_pesquisa")
foc["mes"] = foc["data_pesquisa"].dt.to_period("M")
fm = foc.groupby("mes").tail(1).set_index("mes").sort_index()
fm.index = fm.index.to_timestamp(how="end")
fm = fm[fm.index >= "2012-01-01"]

meta_m = pd.Series([j.METAS_ANUAIS[a][0] for a in fm.index.year],
                   index=fm.index)
tol_m = pd.Series([j.METAS_ANUAIS[a][1] for a in fm.index.year],
                  index=fm.index)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(fm.index, fm["media"], color=AZUL, lw=1.5, label="Focus 12m")
ax1.plot(meta_m.index, meta_m, color=VERMELHO, ls="--", lw=1.2,
         label="meta em vigor")
ax1.plot(meta_m.index, meta_m + tol_m, color=LARANJA, ls=":", lw=1.2,
         label="teto")
ax1.set_title("Expectativa de IPCA 12m (base 0, não suavizada)")
ax1.set_ylabel("%")
ax1.legend(fontsize=9)
ax2.plot(fm.index, fm["desvio_padrao"], color=ROXO, lw=1.2)
ax2.set_title("Dispersão das expectativas (desvio-padrão, p.p.)")
ax2.set_ylabel("p.p.")
fig.tight_layout()
salvar(fig, "09_focus")

fig -> viz/figs/09_focus.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/09_focus.png')

**Figura:** `viz/figs/09_focus.png` — à esquerda, o Focus 12m contra a meta e o teto em vigor; à direita, a dispersão (desvio-padrão) das expectativas.

## 7 · O que as atas dizem (citação verbatim)

As citações são comparadas com o texto bruto das atas; `verbatim_ok=True` significa que a frase aparece exatamente no arquivo. Artefatos de OCR do texto bruto (por exemplo, `El Niño ,`) são preservados na citação.

In [14]:
\
cit = j.citacoes_atas()
display(cit[["num", "data", "tema", "trecho", "verbatim_ok"]])
display(j.termo_por_ata())
for r in cit.itertuples():
    print(f"\nAta {r.num} ({r.data}) — {r.tema}:")
    print(textwrap.fill(f"“{r.trecho}”", width=100,
                        initial_indent="  ", subsequent_indent="  "))

,num,data,tema,trecho,verbatim_ok
0,280,2026-08-05,mercado de trabalho,"A taxa de desemprego mantém-se em patamares historicamente baixos, enquanto o crescimento da taxa de ocupação vem ap...",True
1,280,2026-08-05,salário × produtividade,"Os rendimentos médios desaceleraram na margem, mesmo em termos nominais, mas ainda crescem em termos reais a uma tax...",True
2,281,2026-09-16,mercado de trabalho aquecido,"Em relação ao cenário doméstico, o conjunto dos indicadores divulgados desde a última reunião mostra uma moderação g...",True
3,282,2026-11-04,choques de oferta,"quatro choques de oferta: (i) a elevação dos preços do petróleo, em decorrência dos conflitos no Oriente Médio; (ii)...",True
4,282,2026-11-04,expectativas desancoradas,"em um ambiente de expectativas desancoradas, como é o caso do atual, exige-se uma restrição monetária maior e por ma...",True


,num,data,termo,contem
0,280,2026-08-05,mercado de trabalho aquecido,True
1,281,2026-09-16,mercado de trabalho aquecido,True
2,282,2026-11-04,mercado de trabalho aquecido,False



Ata 280 (2026-08-05) — mercado de trabalho:
  “A taxa de desemprego mantém-se em patamares historicamente baixos, enquanto o crescimento da taxa
  de ocupação vem apresentando desaceleração.”

Ata 280 (2026-08-05) — salário × produtividade:
  “Os rendimentos médios desaceleraram na margem, mesmo em termos nominais, mas ainda crescem em
  termos reais a uma taxa superior à da produtividade do trabalho.”

Ata 281 (2026-09-16) — mercado de trabalho aquecido:
  “Em relação ao cenário doméstico, o conjunto dos indicadores divulgados desde a última reunião
  mostra uma moderação gradual da atividade econômica, principalmente em setores mais cíclicos,
  embora ainda em patamar resiliente e mercado de trabalho aquecido.”

Ata 282 (2026-11-04) — choques de oferta:
  “quatro choques de oferta: (i) a elevação dos preços do petróleo, em decorrência dos conflitos no
  Oriente Médio; (ii) os efeitos climáticos associados ao fenômeno El Niño , com impactos sobre
  produtividade agrícola e custos de 

## 8 · O que o BC diz / o que os dados mostram

Tabela gerada por `justificativas.veredito()`: cada linha contrasta a justificativa com o dado e com a leitura possível. Os números são calculados nas células anteriores, não digitados à mão.

In [15]:
\
v = j.veredito()
with pd.option_context("display.max_colwidth", None):
    display(v[["justificativa_bc", "dado", "leitura", "veredito"]].rename(
        columns={
            "justificativa_bc": "o que o BC diz",
            "dado": "o que os dados mostram",
            "leitura": "leitura",
            "veredito": "veredito",
        }))

,o que o BC diz,o que os dados mostram,leitura,veredito
0,A inflação está acima da meta e exige juro alto.,"IPCA 12m em out/2026: 4.63% contra teto de 4.50%; em 2021, 2022 e 2024 o IPCA ficou acima do teto. Média 2021–25: 5.91% contra meta média de 3.30% (teto médio 4.80%).","O fato (inflação acima da meta) procede. Mas o juro alto não entregou a meta: em 3 dos 5 anos fechados desde 2021 o IPCA ficou acima do teto (2021, 2022 e 2024); 2026 parcial também está acima.",fato procede; eficácia não comprovada
1,O mercado de trabalho está aquecido.,Desemprego 5.2% (09/2026; mínimo 5.1% em 12/2025); nível de ocupação 59.0% (máxima da série); informalidade 37.5% contra 39.9% em 2012–19; participação 62.2% contra 62.8%.,"A descrição procede: desemprego perto do mínimo, ocupação em recorde e informalidade abaixo da média. Mas a participação também está abaixo da média histórica — parte do aperto vem de menor oferta de trabalho, não só de demanda. E mercado apertado não se traduz automaticamente em inflação cheia (ver linha da Phillips).",procede como descrição; não como prova de pressão
2,Os rendimentos crescem acima da produtividade.,Salário real +15.5% e produtividade +0.6% entre 2012 e 2025 (razão 1.15); ULC de +5.2 p.p. em 2025.,"Procede em 2023–25, mas com duas ressalvas: o salário real caiu 8,4% entre 2020 e 2022 e a produtividade é por trabalhador, não por hora. O gap acumulado é de ~15% em 13 anos (~1,1% ao ano), compatível com recuperação, não com espiral.",procede em parte (período e magnitude)
3,Desemprego baixo pressiona a inflação e exige juro alto.,"Phillips do IPCA cheio: coef. do desemprego -0.05 (p=0.73, R²=0.00); com Focus e câmbio, +0.20 (p=0.08, R²=0.59). Em 2021–26 o sinal é positivo (+0.53, p=0.01). Serviços com controles: -0.31 (p=0.001).","Não se sustenta para o IPCA cheio. Para serviços há associação negativa, mas é correlação, não causalidade: a desinflação de 2022–26 ocorreu com o desemprego caindo de 9.5% (média de 2022) para 5.2% (09/2026).",não se sustenta como explicação principal
4,As expectativas estão desancoradas e exigem juro maior por mais tempo.,"Focus de 13/11/2026: 2026 5.11%, 2030 3.86%; 12m 5.15%. Expectativa acima da meta em vigor em 98.6% dos meses desde 2021. Gap médio à meta: 0.77 p.p. em 2012–19 vs 1.38 p.p. em 2021–26; dispersão 0.38 → 0.46 p.p.","O diagnóstico procede: todas as projeções estão acima da meta; 2026 e 2027 também acima do teto de 4,5%. Mas o Focus não identifica a causa — pode refletir choques de oferta, risco fiscal ou a própria meta.",procede como diagnóstico; causa não identificada
5,Choques de oferta explicam a inflação.,"Ata 282 (04/11/2026) lista quatro choques: petróleo, El Niño, fim da escala 6X1 e depreciação do câmbio.",Procede. O próprio BC desloca a explicação para a oferta e não repete 'mercado de trabalho aquecido' na ata 282; o termo aparece nas atas 280 e 281.,procede


**Achados verificáveis.**

1. **A inflação acima da meta é fato — mas o juro alto não a entregou.**
   Com as metas efetivas, o IPCA furou o teto em **2015, 2021, 2022, 2024**
   e em **2026** (parcial); ficou dentro em 2012–14, 2016, 2018–20, 2023 e
   2025, e **abaixo do piso em 2017**. Desde 2021, foram **3 anos fechados
   acima do teto em 5** (2021, 2022, 2024), e o 12m de out/26 (4,63%) segue
   acima de 4,50%. Isso não prova que o juro alto "falhou" — faltam
   contrafactual e choques externos —, mas também não mostra que ele
   entregou a meta.

2. **A composição do IPCA não é a de uma economia "superaquecida" por
   salários.** Em 2021, transportes (4,61 p.p.) e habitação (2,11 p.p.)
   explicam a maior parte; em 2022, alimentação (2,54) e saúde (1,48); em
   2024, alimentação (1,66) e saúde (0,82); em 2026, alimentação (1,23) e
   saúde (0,74). O proxy de serviços (grupos 6–9) responde por 11,5% do IPCA
   em 2021 e 39–46% em 2022–26 — não é o motor principal em 2021. A série
   oficial de serviços (SGS 10844) está na tabela ao lado do proxy.

3. **A curva de Phillips não aparece no IPCA cheio.** Na amostra 2012–26,
   o coeficiente do desemprego é −0,05 (p=0,73, R²=0,00); com Focus e câmbio,
   +0,20 (p=0,08). Para serviços, o sinal é negativo e significativo (−0,50,
   p=0,001; com controles, −0,31, p=0,001). Em 2021–26, o IPCA cheio tem
   sinal **positivo** (+0,53, p=0,01). Isso é associação condicional, não
   causalidade: a desinflação de 2022–26 ocorreu com o desemprego caindo de
   9,5% (média de 2022) para 5,2% (set/26).

4. **Salário acima da produtividade: procede em parte.** Salário real
   +15,5% contra produtividade +0,6% entre 2012 e 2025 (razão 1,15); a ULC
   foi positiva em 2023–25 (+5,3, +3,1, +5,2 p.p.). Mas o salário real caiu
   8,4% entre 2020 e 2022, e a produtividade é por trabalhador, não por
   hora; o gap acumulado é compatível com recuperação, não com espiral.

5. **Mercado de trabalho apertado: descrição procede, prova de pressão não.**
   Desemprego 5,2% (mínimo 5,1% em dez/25), ocupação 59,0% (recorde),
   informalidade 37,5% (abaixo de 39,9% em 2012–19) — mas participação 62,2%,
   abaixo de 62,8% em 2012–19. Parte do aperto vem de menor oferta de
   trabalho. E o mercado apertado não se traduz automaticamente em IPCA
   cheio (ver item 3).

6. **Expectativas desancoradas: diagnóstico procede; causa não identificada.**
   Focus de 13/11/2026: 2026 5,11%, 2027 4,96%, 2028 4,05%, 2029 3,89%,
   2030 3,86%; 12m 5,15%. O nível médio do Focus 12m **caiu** de 5,24%
   (2012–19) para 4,63% (2021–26), mas a meta média caiu mais (4,47% →
   3,25%): o **gap à meta subiu de 0,77 para 1,38 p.p.** e a fração de meses
   acima do teto **triplicou** (11,5% → 39,4%). Em 2021–26, a expectativa
   ficou acima da meta em vigor em 98,6% dos meses. O Focus é expectativa de
   mercado, não medida de credibilidade, e não diz *por que* está
   desancorado.

7. **As atas confirmam o deslocamento para a oferta.** As atas 280 (05/08/26)
   e 281 (16/09/26) dizem "mercado de trabalho aquecido"; a 282 (04/11/26)
   lista **quatro choques de oferta** (petróleo, El Niño, fim da escala 6X1,
   câmbio) e fala em "expectativas desancoradas", sem repetir "aquecido".

**Limites declarados.** Não há identificação causal: a Phillips é
correlação condicional; a decomposição é contábil (soma dos grupos ≈ IPCA,
resíduo ≤ 0,36 p.p.); a produtividade é por trabalhador (a PNAD agregada não
publica horas no SIDRA); o proxy de serviços (grupos 6–9) não é a série
oficial SGS 10844; 2026 está incompleto (IPCA até out, PNAD até o 3º tri);
o Focus é expectativa de mercado, não credibilidade do BC. Onde a série não
permite separar os mecanismos, a leitura é "não verificável", não "falsa".